In [1]:
import onnx
from onnx import shape_inference
import onnx_graphsurgeon as gs

model = onnx.load("models/0kc5po4ee18_int8_smoothquant_onnx_cuda_enc.onnx")
onnx.checker.check_model(model)

In [2]:
INT8_TYPES = {onnx.TensorProto.INT8, onnx.TensorProto.UINT8}

def build_dtype_map(model: onnx.ModelProto):
    """tensor_name -> elem_type (onnx.TensorProto.DataType) when available."""
    m = {}
    g = model.graph

    def add_vi(vi):
        tt = vi.type.tensor_type
        if tt is None:
            return
        if tt.elem_type != 0:  # 0 means 'unknown'
            m[vi.name] = tt.elem_type

    # Graph inputs / outputs / value_info
    for vi in list(g.input) + list(g.output) + list(g.value_info):
        add_vi(vi)

    # Initializers (weights) have dtypes
    for init in g.initializer:
        m[init.name] = init.data_type

    return m

def find_last_int8_node(model: onnx.ModelProto):
    # Best effort to fill in missing value_info types
    try:
        model = shape_inference.infer_shapes(model)
    except Exception as e:
        print("[warn] shape inference failed:", e)

    dtype_map = build_dtype_map(model)

    last = None
    for idx, node in enumerate(model.graph.node):
        out_types = []
        for out in node.output:
            t = dtype_map.get(out, None)
            out_types.append(t)
        if any(t in INT8_TYPES for t in out_types if t is not None):
            last = (idx, node, out_types)

    return model, dtype_map, last

model, dtype_map, last = find_last_int8_node(model)

In [5]:
idx, node, out_types = last

In [6]:
def tname(t):
    if t is None:
        return "UNKNOWN"
    return onnx.TensorProto.DataType.Name(t)
print("=== Last INT8-producing node ===")
print("Index:", idx)
print("Name:", node.name if node.name else "(no name)")
print("Op:", node.op_type)
print("Inputs:", list(node.input))
print("Outputs:", list(node.output))
print("Output dtypes:", [tname(t) for t in out_types])

=== Last INT8-producing node ===
Index: 2012
Name: /enc_post/enc_post.0/blocks.5/blocks.5.1/mlp/fc2/weight_quantizer/QuantizeLinear
Op: QuantizeLinear
Inputs: ['enc_post.0.blocks.5.1.mlp.fc2.weight', '/enc_post/enc_post.0/blocks.5/blocks.5.1/mlp/fc2/weight_quantizer/Constant_1_output_0', '/enc_post/enc_post.0/blocks.5/blocks.5.1/mlp/fc2/weight_quantizer/Constant_output_0']
Outputs: ['/enc_post/enc_post.0/blocks.5/blocks.5.1/mlp/fc2/weight_quantizer/QuantizeLinear_output_0']
Output dtypes: ['INT8']
